# GPT 2.0 구현 — Tiny Character-Level GPT

이 노트북은 강의안 `notebook_04`, `notebook_05`, `notebook_06`의 내용을 통합하여  
GPT-2의 핵심인 **decoder-only Transformer language model**을 구현합니다.

구현 요소:

- GPT-style next-token dataset
- token embedding과 positional embedding
- masked self-attention
- multi-head attention
- feedforward network
- residual connection
- layer normalization
- next-token prediction
- text generation

In [1]:
import json
import random
import urllib.request
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader

from model import TinyGPT

torch.manual_seed(42)
random.seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cpu


## 1. 데이터 불러오기

Alice in Wonderland 데이터를 사용합니다.  
각 문자를 하나의 token으로 취급하는 character-level language model입니다.

In [2]:
DATA_URL = (
    "https://raw.githubusercontent.com/karpathy/"
    "char-rnn/master/data/tinyshakespeare/input.txt"
)

data_path = Path("input.txt")

if not data_path.exists():
    urllib.request.urlretrieve(DATA_URL, data_path)

text = data_path.read_text(encoding="utf-8")

print("number of characters:", len(text))
print(text[:500])

number of characters: 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


## 2. Character-level tokenization

등장하는 모든 문자를 vocabulary로 만들고 정수 index로 변환합니다.

In [3]:
characters = sorted(set(text))
vocab_size = len(characters)

stoi = {
    character: index
    for index, character in enumerate(characters)
}

itos = {
    index: character
    for character, index in stoi.items()
}

def encode(string):
    return [stoi[character] for character in string]

def decode(indices):
    return "".join(itos[int(index)] for index in indices)

data = torch.tensor(encode(text), dtype=torch.long)

split_index = int(0.9 * len(data))
train_data = data[:split_index]
validation_data = data[split_index:]

print("vocabulary size:", vocab_size)
print("train tokens:", len(train_data))
print("validation tokens:", len(validation_data))

vocabulary size: 65
train tokens: 1003854
validation tokens: 111540


## 3. GPT-style next-token dataset

입력 `x`의 각 위치에 대한 정답 `y`는 바로 다음 문자입니다.

In [4]:
class NextTokenDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, index):
        x = self.data[index : index + self.block_size]
        y = self.data[index + 1 : index + self.block_size + 1]
        return x, y


block_size = 64
batch_size = 64

train_dataset = NextTokenDataset(train_data, block_size)
validation_dataset = NextTokenDataset(validation_data, block_size)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=batch_size,
    shuffle=False,
    drop_last=True,
)

x_batch, y_batch = next(iter(train_loader))

print("input shape:", x_batch.shape)
print("target shape:", y_batch.shape)
print("input example:", decode(x_batch[0].tolist()))
print("target example:", decode(y_batch[0].tolist()))

input shape: torch.Size([64, 64])
target shape: torch.Size([64, 64])
input example: 
With her companion grief must end her life.

JOHN OF GAUNT:
Sis
target example: With her companion grief must end her life.

JOHN OF GAUNT:
Sist


## 4. Tiny GPT 모델 생성

`model.py`에 다음 요소가 구현되어 있습니다.

1. masked self-attention head  
2. multi-head attention  
3. feedforward network  
4. residual connection  
5. layer normalization  
6. 여러 Transformer decoder block  
7. language-model head

In [5]:
embedding_dim = 128
num_heads = 4
num_layers = 4
dropout = 0.1

model = TinyGPT(
    vocab_size=vocab_size,
    block_size=block_size,
    emb_dim=embedding_dim,
    num_heads=num_heads,
    num_layers=num_layers,
    dropout=dropout,
).to(device)

number_of_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print(f"number of parameters: {number_of_parameters:,}")

number of parameters: 816,705


## 5. 초기 loss 확인

학습 전에는 모델의 예측이 무작위에 가깝기 때문에 loss가 높습니다.

In [6]:
x_batch = x_batch.to(device)
y_batch = y_batch.to(device)

logits, initial_loss = model(x_batch, y_batch)

print("logits shape:", logits.shape)
print("initial loss:", initial_loss.item())

logits shape: torch.Size([64, 64, 65])
initial loss: 4.3389892578125


## 6. 학습 및 검증

기본 설정은 10 epoch, epoch당 최대 300 step입니다.

실행 시간이 너무 길면 먼저:

```python
epochs = 1
max_steps_per_epoch = 20
```

으로 작동 여부를 확인한 뒤 다시 늘리면 됩니다.

In [7]:
@torch.no_grad()
def estimate_loss(model, loader, device, max_batches=20):
    model.eval()
    losses = []

    for batch_index, (x, y) in enumerate(loader):
        if batch_index >= max_batches:
            break

        x = x.to(device)
        y = y.to(device)

        _, loss = model(x, y)
        losses.append(loss.item())

    model.train()
    return sum(losses) / max(len(losses), 1)


learning_rate = 3e-4
epochs = 10
max_steps_per_epoch = 300

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
)

training_history = []

for epoch in range(1, epochs + 1):
    model.train()
    running_loss = 0.0
    completed_steps = 0

    for step, (x, y) in enumerate(train_loader, start=1):
        if step > max_steps_per_epoch:
            break

        x = x.to(device)
        y = y.to(device)

        _, loss = model(x, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        completed_steps += 1

    train_loss = estimate_loss(
        model,
        train_loader,
        device,
    )

    validation_loss = estimate_loss(
        model,
        validation_loader,
        device,
    )

    result = {
        "epoch": epoch,
        "training_batch_loss": running_loss / max(completed_steps, 1),
        "train_loss": train_loss,
        "validation_loss": validation_loss,
    }

    training_history.append(result)

    print(
        f"epoch {epoch:02d} | "
        f"train loss {train_loss:.4f} | "
        f"validation loss {validation_loss:.4f}"
    )

epoch 01 | train loss 2.3722 | validation loss 2.4345
epoch 02 | train loss 2.1238 | validation loss 2.1917
epoch 03 | train loss 1.9693 | validation loss 2.0607
epoch 04 | train loss 1.8515 | validation loss 1.9660
epoch 05 | train loss 1.7771 | validation loss 1.8835
epoch 06 | train loss 1.7179 | validation loss 1.8142
epoch 07 | train loss 1.6661 | validation loss 1.7519


KeyboardInterrupt: 

## 7. 모델 저장

학습한 parameter와 vocabulary 정보를 함께 저장합니다.

In [ ]:
checkpoint = {
    "model_state_dict": model.state_dict(),
    "config": {
        "vocab_size": vocab_size,
        "block_size": block_size,
        "emb_dim": embedding_dim,
        "num_heads": num_heads,
        "num_layers": num_layers,
        "dropout": dropout,
    },
    "stoi": stoi,
    "itos": {
        str(index): character
        for index, character in itos.items()
    },
}

torch.save(
    checkpoint,
    "tiny_gpt_checkpoint.pt",
)

Path("training_history.json").write_text(
    json.dumps(training_history, indent=2),
    encoding="utf-8",
)

print("model and training history saved")

## 8. 학습된 language model로 텍스트 예측·생성

`Alice`를 시작 문장으로 주고 다음 문자를 반복적으로 예측합니다.

- `temperature`가 낮을수록 안정적인 출력
- `top_k`는 확률이 높은 후보 일부만 사용

In [ ]:
prompt = "Alice"

prompt_indices = [
    stoi[character]
    for character in prompt
    if character in stoi
]

context = torch.tensor(
    [prompt_indices],
    dtype=torch.long,
    device=device,
)

generated_indices = model.generate(
    context,
    max_new_tokens=500,
    temperature=0.8,
    top_k=40,
)

generated_text = decode(
    generated_indices[0].tolist()
)

print(generated_text)

Path("generated_text.txt").write_text(
    generated_text,
    encoding="utf-8",
)

print("\ngenerated_text.txt saved")

## 9. 결과 해석

학습이 진행되면 다음과 같은 패턴이 나타납니다.

- 등장인물 이름 뒤에 `:`가 붙는 형식
- 줄바꿈과 문장 부호
- 영어 단어와 비슷한 문자열
- Alice in Wonderland 희곡 대사와 유사한 구조

이 모델은 작은 character-level GPT이므로 완벽한 문법이나 의미를 생성하지는 않습니다.  
그러나 masked self-attention을 이용해 이전 문자 문맥을 바탕으로 다음 문자 확률을 학습합니다.